# Returns Summary Analysis
## Processing Daily Returns Data for All Months

This notebook processes daily returns data from January to April 2026 and creates:
1. **Monthly_Summary** - All customers with brand columns (FRESH/EXP) by month
2. **Total_All_Months** - All customers with totals across all months
3. **Route_Summary** - All routes with brand columns and totals

**NOTE:** All records are included, even those with zero returns or empty routes.

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## Configuration

In [35]:
# File paths
DATA_DIR = Path('data/mehul-returns')
OUTPUT_DIR = Path('reports/september/wk2')
OUTPUT_DIR.mkdir(exist_ok=True)
SALES_DIR = Path("sales")

# Input files
files = {
    'January': DATA_DIR / 'JAN_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES JANUARY 2026.xlsx',
    'February': DATA_DIR / 'FEB_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES FEBRUARY 2026 FINAL.xlsx',
    'March': DATA_DIR / 'MARCH_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES MARCH 2026 FINAL.xlsx',
    'April': DATA_DIR / 'APRIL_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES APRIL 2026.xlsx',
    'May': DATA_DIR / 'MAY_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES MAY 2026.xlsx',
    'June': DATA_DIR / 'JUNE_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES JUNE 2026.xlsx',
    'July': DATA_DIR / 'JULY_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES JULY 2026.xlsx',
    'August': DATA_DIR / 'AUGUST_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES AUGUST 2026.xlsx',
    'September': DATA_DIR / 'SEPTEMBER_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES SEPTEMBER 2026.xlsx',
    'October': DATA_DIR / 'OCTOBER_DAILY RETURNS FOR DEPOTS , SUPERMARKETS & VAN SALES OCTOBER 2026.xlsx'
}

# '50/50'
# Brand names as they appear in the data
BRANDS = ['P', '4SQ', 'B', 'S', '200G', 'O.P', '800G', '600G', 'MANDO', 'FAMILY PACK', 'MAICHO', 'XLOONG', 'BB']


def add_return_totals(dataframe):
    """Add fresh, expired, and overall totals across all tracked brands."""
    fresh_columns = [f'{brand}_FRESH' for brand in BRANDS]
    exp_columns = [f'{brand}_EXP' for brand in BRANDS]
    dataframe['TOTAL_FRE'] = dataframe[fresh_columns].sum(axis=1)
    dataframe['TOTAL_EXP'] = dataframe[exp_columns].sum(axis=1)
    dataframe['TOTAL_ROUTE'] = dataframe['TOTAL_FRE'] + dataframe['TOTAL_EXP']
    return dataframe


print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Number of months: {len(files)}")
print(f"Number of brands: {len(BRANDS)}")

Data directory: data\mehul-returns
Output directory: reports\september\wk2
Number of months: 10
Number of brands: 13


In [36]:
# august = pd.ExcelFile(files['August'])
# august.sheet_names

In [37]:
# august = pd.read_excel(files['August'], sheet_name='2ND', skiprows=2)
# august.head()

In [38]:
# max_code, min_code = august['CODE'].max(), august['CODE'].min()
# max_code, min_code

## Helper Functions

In [39]:
def parse_daily_sheet(file_path, sheet_name):
    """Parse one daily returns sheet using its two-row brand/type header."""
    try:
        raw_df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        if raw_df.shape[0] < 3 or raw_df.shape[1] < 5:
            return None

        base_columns = ['ROUTE_NO', 'ROUTE', 'CATEGORY', 'CODE', 'NAME']
        brands_row = raw_df.iloc[1, 5:].tolist()
        types_row = raw_df.iloc[2, 5:].tolist()
        source_columns = []
        current_brand = None
        used_columns = set()

        for position, (brand, col_type) in enumerate(zip(brands_row, types_row), start=5):
            has_brand = pd.notna(brand) and str(brand).strip()
            if has_brand:
                current_brand = str(brand).strip().upper()

            type_name = str(col_type).strip().upper() if pd.notna(col_type) else ''
            type_name = {'EX': 'EXP', '1': 'EXP', '1.0': 'EXP'}.get(type_name, type_name)
            if current_brand == 'TOTAL' and (has_brand or type_name):
                column_name = 'TOTAL_ROUTE'
            elif current_brand and type_name in {'FRESH', 'EXP'}:
                column_name = f'{current_brand}_{type_name}'
            else:
                column_name = f'_IGNORED_{position}'

            if column_name != 'TOTAL_ROUTE' and column_name in used_columns:
                column_name = f'{column_name}_{position}'
            used_columns.add(column_name)
            source_columns.append(column_name)

        data_df = raw_df.iloc[3:].copy()
        data_df.columns = base_columns + source_columns
        data_df = data_df.dropna(how='all')
        data_df = data_df[data_df['NAME'].astype('string').str.strip().str.upper().ne('TOTAL')]
        data_df['ROUTE'] = data_df['ROUTE'].fillna('')
        numeric_columns = [
            column for column in data_df.columns
            if column == 'TOTAL_ROUTE' or any(
                column == f'{brand}_{suffix}'
                for brand in BRANDS
                for suffix in ('FRESH', 'EXP')
            )
        ]
        for column in numeric_columns:
            data_df[column] = pd.to_numeric(data_df[column], errors='coerce').fillna(0)
        return data_df
    except Exception as error:
        print(f'Error parsing {sheet_name}: {error}')
        return None


def normalize_customer_fields(data_df):
    """Normalize route and customer fields before aggregating daily records."""
    text_cols = ['ROUTE', 'CATEGORY', 'NAME']
    for col in text_cols:
        data_df[col] = data_df[col].astype('string').fillna('').str.strip()
        data_df[col] = data_df[col].replace({'<NA>': '', 'nan': ''})

    data_df['ROUTE_NO'] = pd.to_numeric(data_df['ROUTE_NO'], errors='coerce')
    data_df['CODE'] = pd.to_numeric(data_df['CODE'], errors='coerce')

    route_lookup = (
        data_df.loc[data_df['ROUTE'].ne('') & data_df['ROUTE_NO'].notna(), ['ROUTE', 'ROUTE_NO']]
        .groupby(['ROUTE', 'ROUTE_NO'], as_index=False)
        .size()
        .sort_values(['ROUTE', 'size', 'ROUTE_NO'], ascending=[True, False, True])
        .drop_duplicates('ROUTE')
        .rename(columns={'ROUTE_NO': 'INFERRED_ROUTE_NO'})
        [['ROUTE', 'INFERRED_ROUTE_NO']]
    )
    data_df = data_df.merge(route_lookup, on='ROUTE', how='left')
    data_df['ROUTE_NO'] = data_df['INFERRED_ROUTE_NO'].fillna(data_df['ROUTE_NO'])
    data_df = data_df.drop(columns='INFERRED_ROUTE_NO')

    code_lookup = (
        data_df.loc[data_df['ROUTE'].ne('') & data_df['NAME'].ne('') & data_df['CODE'].notna(), ['ROUTE', 'NAME', 'CODE']]
        .groupby(['ROUTE', 'NAME'])['CODE']
        .agg(lambda values: values.iloc[0] if values.nunique() == 1 else pd.NA)
        .rename('INFERRED_CODE')
        .reset_index()
    )
    data_df = data_df.merge(code_lookup, on=['ROUTE', 'NAME'], how='left')
    data_df['CODE'] = data_df['CODE'].fillna(data_df['INFERRED_CODE'])
    data_df = data_df.drop(columns='INFERRED_CODE')

    customer_mask = data_df['ROUTE'].ne('') | data_df['NAME'].ne('') | data_df['CODE'].notna()
    return data_df.loc[customer_mask].copy()


def aggregate_daily_sheets(file_path, month_name):
    """Aggregate all daily sheets from a month file."""
    print(f'\nProcessing {month_name}...')
    xl = pd.ExcelFile(file_path)
    import re
    daily_sheets = [s for s in xl.sheet_names if any(day in s.upper() for day in
                    ['1ST', '2ND', '3RD', '4TH', '5TH', '6TH', '7TH', '8TH', '9TH',
                     '10TH', '11TH', '12TH', '13TH', '14TH', '15TH', '16TH', '17TH',
                     '18TH', '19TH', '20TH', '21ST', '22ND', '23RD', '24TH', '25TH',
                     '26TH', '27TH', '28TH', '29TH', '30TH', '31ST']) or re.match(r'^SHEET\d+$', s.upper())]
    print(f'  Found {len(daily_sheets)} daily sheets')
    all_data = []
    for sheet in daily_sheets:
        df = parse_daily_sheet(file_path, sheet)
        if df is not None and len(df) > 0:
            all_data.append(df)
    if not all_data:
        print(f'  No valid data found for {month_name}')
        return None
    combined = normalize_customer_fields(pd.concat(all_data, ignore_index=True))
    group_cols = ['ROUTE_NO', 'ROUTE', 'CATEGORY', 'CODE', 'NAME']
    brand_cols = [
        column for column in combined.columns
        if any(column == f'{brand}_{suffix}' for brand in BRANDS for suffix in ('FRESH', 'EXP'))
    ]
    agg_dict = {column: 'sum' for column in brand_cols}
    if 'TOTAL_ROUTE' in combined.columns:
        agg_dict['TOTAL_ROUTE'] = 'sum'
    monthly_total = combined.groupby(group_cols, as_index=False, dropna=False).agg(agg_dict)
    print(f'  Processed {len(monthly_total)} unique customers/routes')
    return monthly_total


def create_standard_columns():
    """Create the standard column structure with all brands and FRESH/EXP."""
    base_cols = ['ROUTE_NO', 'ROUTE', 'CATEGORY', 'CODE', 'NAME']
    brand_cols = []
    for brand in BRANDS:
        brand_cols.extend([f'{brand}_FRESH', f'{brand}_EXP'])
    return base_cols + brand_cols + ['TOTAL_ROUTE']


def standardize_dataframe(df, standard_cols):
    """Ensure dataframe has all standard columns, adding missing ones with 0s."""
    for col in standard_cols:
        if col not in df.columns:
            if col in ['ROUTE_NO', 'ROUTE', 'CATEGORY', 'CODE', 'NAME']:
                df[col] = ''
            else:
                df[col] = 0
    return df[standard_cols]


print('Helper functions defined successfully!')

Helper functions defined successfully!


## Step 1: Load and Process All Monthly Data

In [40]:
# Create standard column structure
standard_columns = create_standard_columns()
standard_columns[-1:-1] = ['TOTAL_FRE', 'TOTAL_EXP']
print(f"Standard columns created: {len(standard_columns)} columns")
print(f"\nColumn structure:")
print("  Base columns:", standard_columns[:5])
print(f"  Brand columns: {len(standard_columns[5:-3])} columns (FRESH/EXP for each brand)")
print("  Return totals:", standard_columns[-3:-1])
print("  Total column:", standard_columns[-1])

Standard columns created: 34 columns

Column structure:
  Base columns: ['ROUTE_NO', 'ROUTE', 'CATEGORY', 'CODE', 'NAME']
  Brand columns: 26 columns (FRESH/EXP for each brand)
  Return totals: ['TOTAL_FRE', 'TOTAL_EXP']
  Total column: TOTAL_ROUTE


In [41]:
# Process each month
monthly_data = {}

for month_name, file_path in files.items():
    if file_path.exists():
        df = aggregate_daily_sheets(file_path, month_name)
        if df is not None:
            # Standardize columns and calculate fresh/expired totals
            df_standard = add_return_totals(standardize_dataframe(df, standard_columns))
            monthly_data[month_name] = df_standard
            print(f"  ✓ {month_name}: {len(df_standard)} records processed")
    else:
        print(f"  ✗ {month_name}: File not found")

print(f"\n{'='*60}")
print(f"Total months processed: {len(monthly_data)}")


Processing January...
  Found 53 daily sheets
  Processed 819 unique customers/routes
  ✓ January: 819 records processed

Processing February...
  Found 54 daily sheets
  Processed 838 unique customers/routes
  ✓ February: 838 records processed

Processing March...
  Found 57 daily sheets
  Processed 875 unique customers/routes
  ✓ March: 875 records processed

Processing April...
  Found 56 daily sheets
  Processed 870 unique customers/routes
  ✓ April: 870 records processed

Processing May...
  Found 57 daily sheets
  Processed 928 unique customers/routes
  ✓ May: 928 records processed

Processing June...
  Found 56 daily sheets
  Processed 954 unique customers/routes
  ✓ June: 954 records processed

Processing July...
  Found 57 daily sheets
  Processed 971 unique customers/routes
  ✓ July: 971 records processed

Processing August...
  Found 57 daily sheets
  Processed 995 unique customers/routes
  ✓ August: 995 records processed

Processing September...
  Found 56 daily sheets
Err

## Step 2: Create Monthly Summary Sheets

In [42]:
# Create separate sheets for each month
monthly_summary_sheets = {}

for month_name, df in monthly_data.items():
    # Sort by ROUTE, then CATEGORY, then NAME
    df_sorted = df.sort_values(['ROUTE', 'CATEGORY', 'NAME']).reset_index(drop=True)

    # Recalculate totals for the monthly output
    df_sorted = add_return_totals(df_sorted)

    monthly_summary_sheets[f"{month_name}_Summary"] = df_sorted

    print(f"{month_name}: {len(df_sorted)} customers/routes")
    print(f"  Categories: {df_sorted['CATEGORY'].unique().tolist()}")
    print(f"  Total returns: {df_sorted['TOTAL_ROUTE'].sum():,.0f}")
    print()

January: 819 customers/routes
  Categories: ['DIST', 'DSUPMK', 'INST', 'VAN']
  Total returns: 81,437

February: 838 customers/routes
  Categories: ['DIST', 'DSUPMK', 'INST', 'VAN']
  Total returns: 77,421

March: 875 customers/routes
  Categories: ['DIST', 'DSUPMK', 'INST', 'VAN']
  Total returns: 79,072

April: 870 customers/routes
  Categories: ['DIST', 'DSUPMK', 'INST', 'VAN', '']
  Total returns: 88,712

May: 928 customers/routes
  Categories: ['', 'DIST', 'DSUPMK', 'INST', 'VAN']
  Total returns: 92,327

June: 954 customers/routes
  Categories: ['DSUPMK', 'DIST', 'INST', 'VAN', '']
  Total returns: 88,679

July: 971 customers/routes
  Categories: ['DSUPMK', 'DIST', 'INST', 'VAN', '']
  Total returns: 101,277

August: 995 customers/routes
  Categories: ['DSUPMK', 'DIST', 'INST', 'VAN', '']
  Total returns: 98,586

September: 983 customers/routes
  Categories: ['DIST', 'DSUPMK', 'INST', 'VAN', '']
  Total returns: 49,563



## Step 3: Create Total All Months Summary

In [43]:
# Combine all months and aggregate
all_months_combined = pd.concat(monthly_data.values(), ignore_index=True)
all_months_combined = normalize_customer_fields(all_months_combined)

# Group by customer and sum each brand separately before recalculating totals
# so customers with incomplete route metadata are not discarded.
group_cols = ['ROUTE_NO', 'ROUTE', 'CATEGORY', 'CODE', 'NAME']
brand_data_cols = [f'{brand}_{suffix}' for brand in BRANDS for suffix in ('FRESH', 'EXP')]
summary_cols = brand_data_cols + ['TOTAL_FRE', 'TOTAL_EXP', 'TOTAL_ROUTE']

total_all_months = all_months_combined.groupby(group_cols, as_index=False, dropna=False)[summary_cols].sum()
total_all_months = add_return_totals(total_all_months)

# Sort
total_all_months = total_all_months.sort_values(['ROUTE', 'CATEGORY', 'NAME']).reset_index(drop=True)

print(f"Total All Months Summary:")
print(f"  Total unique customers/routes: {len(total_all_months)}")
print(f"  Total returns (all months): {total_all_months['TOTAL_ROUTE'].sum():,.0f}")
print(f"  Categories: {total_all_months['CATEGORY'].unique().tolist()}")

Total All Months Summary:
  Total unique customers/routes: 1055
  Total returns (all months): 757,074
  Categories: ['', 'DSUPMK', 'DIST', 'INST', 'VAN']


## Step 4: Create Route Summary

In [44]:
# Aggregate by route (sum all customers in each route), retaining blank route fields
route_summary = total_all_months.groupby(['ROUTE_NO', 'ROUTE'], as_index=False, dropna=False)[summary_cols].sum()
route_summary = add_return_totals(route_summary)

# Convert ROUTE_NO to numeric for sorting (handles mixed types)
route_summary['ROUTE_NO'] = pd.to_numeric(route_summary['ROUTE_NO'], errors='coerce')

# Sort by route number
route_summary = route_summary.sort_values('ROUTE_NO', na_position='last').reset_index(drop=True)

# Add a category column (aggregate from customers)
route_categories = total_all_months.groupby('ROUTE', dropna=False)['CATEGORY'].apply(lambda x: ', '.join(x.dropna().unique())).to_dict()
route_summary['CATEGORY'] = route_summary['ROUTE'].map(route_categories)

# Reorder columns
route_cols = ['ROUTE_NO', 'ROUTE', 'CATEGORY'] + summary_cols
route_summary = route_summary[route_cols]

print(f"Route Summary:")
print(f"  Total routes: {len(route_summary)}")
print(f"  Total returns (all routes): {route_summary['TOTAL_ROUTE'].sum():,.0f}")

Route Summary:
  Total routes: 122
  Total returns (all routes): 757,074


## Step 5: Create Brand Summary Statistics

In [45]:
# Calculate totals by brand
brand_summary = []

for brand in BRANDS:
    fresh_col = f"{brand}_FRESH"
    exp_col = f"{brand}_EXP"
    
    fresh_total = total_all_months[fresh_col].sum() if fresh_col in total_all_months.columns else 0
    exp_total = total_all_months[exp_col].sum() if exp_col in total_all_months.columns else 0
    
    brand_summary.append({
        'BRAND': brand,
        'FRESH': fresh_total,
        'EXP': exp_total,
        'TOTAL': fresh_total + exp_total
    })

brand_summary_df = pd.DataFrame(brand_summary)
brand_summary_df = brand_summary_df.sort_values('TOTAL', ascending=False).reset_index(drop=True)

print("\nBrand Summary (All Months):")
print(brand_summary_df.to_string(index=False))
print(f"\nGrand Total: {brand_summary_df['TOTAL'].sum():,.0f}")


Brand Summary (All Months):
      BRAND   FRESH      EXP    TOTAL
          S 66467.0 177548.0 244015.0
          B 32609.0 101857.0 134466.0
       200G 31906.0  74414.0 106320.0
       600G 15951.0  79417.0  95368.0
       800G  7093.0  64657.0  71750.0
        O.P  7533.0  30840.0  38373.0
        4SQ  4406.0  10785.0  15191.0
     MAICHO  1578.0  12394.0  13972.0
      MANDO  2147.0  11402.0  13549.0
     XLOONG  3056.0   9205.0  12261.0
          P  1960.0   8519.0  10479.0
FAMILY PACK   640.0    605.0   1245.0
         BB    84.0      1.0     85.0

Grand Total: 757,074


## Step 6: Export to Excel

In [46]:
# Create output file
output_file = OUTPUT_DIR / 'Returns_Summary_oct_wk2_2026.xlsx'



with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    
    # Write monthly summaries
    for sheet_name, df in monthly_summary_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"✓ Wrote {sheet_name}: {len(df)} rows")
    
    # Write Total All Months
    total_all_months.to_excel(writer, sheet_name='Total_All_Months', index=False)
    print(f"✓ Wrote Total_All_Months: {len(total_all_months)} rows")
    
    # Write Route Summary
    route_summary.to_excel(writer, sheet_name='Route_Summary', index=False)
    print(f"✓ Wrote Route_Summary: {len(route_summary)} rows")
    
    # Write Brand Summary
    brand_summary_df.to_excel(writer, sheet_name='Brand_Summary', index=False)
    print(f"✓ Wrote Brand_Summary: {len(brand_summary_df)} rows")

print(f"\n{'='*60}")
print(f"SUCCESS! File saved to: {output_file}")
print(f"{'='*60}")

✓ Wrote January_Summary: 819 rows
✓ Wrote February_Summary: 838 rows
✓ Wrote March_Summary: 875 rows
✓ Wrote April_Summary: 870 rows
✓ Wrote May_Summary: 928 rows
✓ Wrote June_Summary: 954 rows
✓ Wrote July_Summary: 971 rows
✓ Wrote August_Summary: 995 rows
✓ Wrote September_Summary: 983 rows
✓ Wrote Total_All_Months: 1055 rows
✓ Wrote Route_Summary: 122 rows
✓ Wrote Brand_Summary: 13 rows

SUCCESS! File saved to: reports\september\wk2\Returns_Summary_oct_wk2_2026.xlsx


## Step 7: Summary Statistics

In [47]:
print("\n" + "="*60)
print("FINAL SUMMARY STATISTICS")
print("="*60)

print(f"\nDATA OVERVIEW:")
print(f"  Months processed: {len(monthly_data)}")
print(f"  Total unique customers: {len(total_all_months)}")
print(f"  Total unique routes: {len(route_summary)}")
print(f"  Total brands tracked: {len(BRANDS)}")

print(f"\nRETURNS SUMMARY:")
print(f"  Grand total returns: {total_all_months['TOTAL_ROUTE'].sum():,.0f} loaves")
print(f"  Average per customer: {total_all_months['TOTAL_ROUTE'].mean():,.1f} loaves")
print(f"  Average per route: {route_summary['TOTAL_ROUTE'].mean():,.1f} loaves")

print(f"\nBY CATEGORY:")
category_totals = total_all_months.groupby('CATEGORY')['TOTAL_ROUTE'].sum().sort_values(ascending=False)
for cat, total in category_totals.items():
    percentage = (total / total_all_months['TOTAL_ROUTE'].sum()) * 100
    print(f"  {cat}: {total:,.0f} loaves ({percentage:.1f}%)")

print(f"\nTOP 5 BRANDS BY RETURNS:")
top_brands = brand_summary_df.head(5)
for idx, row in top_brands.iterrows():
    percentage = (row['TOTAL'] / brand_summary_df['TOTAL'].sum()) * 100
    print(f"  {row['BRAND']}: {row['TOTAL']:,.0f} loaves ({percentage:.1f}%) - FRESH: {row['FRESH']:,.0f}, EXP: {row['EXP']:,.0f}")

print(f"\n OUTPUT FILE:")
print(f"  Location: {output_file}")
print(f"  Sheets created:")
for sheet in monthly_summary_sheets.keys():
    print(f"    - {sheet}")
print(f"    - Total_All_Months")
print(f"    - Route_Summary")
print(f"    - Brand_Summary")

print("\n" + "="*60)
print("✓ ANALYSIS COMPLETE!")
print("="*60)


FINAL SUMMARY STATISTICS

DATA OVERVIEW:
  Months processed: 9
  Total unique customers: 1055
  Total unique routes: 122
  Total brands tracked: 13

RETURNS SUMMARY:
  Grand total returns: 757,074 loaves
  Average per customer: 717.6 loaves
  Average per route: 6,205.5 loaves

BY CATEGORY:
  DIST: 285,498 loaves (37.7%)
  DSUPMK: 279,930 loaves (37.0%)
  VAN: 127,214 loaves (16.8%)
  : 63,609 loaves (8.4%)
  INST: 823 loaves (0.1%)

TOP 5 BRANDS BY RETURNS:
  S: 244,015 loaves (32.2%) - FRESH: 66,467, EXP: 177,548
  B: 134,466 loaves (17.8%) - FRESH: 32,609, EXP: 101,857
  200G: 106,320 loaves (14.0%) - FRESH: 31,906, EXP: 74,414
  600G: 95,368 loaves (12.6%) - FRESH: 15,951, EXP: 79,417
  800G: 71,750 loaves (9.5%) - FRESH: 7,093, EXP: 64,657

 OUTPUT FILE:
  Location: reports\september\wk2\Returns_Summary_oct_wk2_2026.xlsx
  Sheets created:
    - January_Summary
    - February_Summary
    - March_Summary
    - April_Summary
    - May_Summary
    - June_Summary
    - July_Summary
   